## Setup Student Projects

After creating projects for each student ([Instructions](https://github.com/tulane-intro-ai-engineering/main/blob/main/setup/OpenAI-Instructors.md)), the below will set limits on throughput and set alerts for spending.

In [4]:
#!/usr/bin/env python3
import getpass
import os
import sys
import time
import requests

os.environ["OPENAI_ADMIN_KEY"] = getpass.getpass("OpenAI Admin API key: ")

OPENAI_ADMIN_KEY = os.getenv("OPENAI_ADMIN_KEY")
if not OPENAI_ADMIN_KEY:
    print("Missing OPENAI_ADMIN_KEY env var (must be an Admin API key).", file=sys.stderr)
    sys.exit(1)

BASE_URL = "https://api.openai.com/v1"
HEADERS = {
    "Authorization": f"Bearer {OPENAI_ADMIN_KEY}",
    "Content-Type": "application/json",
}

TARGET_MODELS = {"gpt-4o-mini", "gpt-3.5-turbo-0125"}
TARGET_RPM = 120
TARGET_TPM = 100_000

def get_all_projects():
    projects = []
    after = None
    while True:
        params = {"limit": 100}
        if after:
            params["after"] = after
        r = requests.get(f"{BASE_URL}/organization/projects", headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        projects.extend(data.get("data", []))
        if not data.get("has_more"):
            break
        after = data.get("last_id")
    return projects

def get_project_rate_limits(project_id: str):
    r = requests.get(f"{BASE_URL}/organization/projects/{project_id}/rate_limits",
                     headers=HEADERS, params={"limit": 100}, timeout=30)
    r.raise_for_status()
    return r.json().get("data", [])

def update_rate_limit(project_id: str, rate_limit_id: str, rpm: int, tpm: int):
    payload = {
        "max_requests_per_1_minute": rpm,
        "max_tokens_per_1_minute": tpm,
    }
    r = requests.post(f"{BASE_URL}/organization/projects/{project_id}/rate_limits/{rate_limit_id}",
                      headers=HEADERS, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()

def main():
    projects = get_all_projects()
    print(f"Found {len(projects)} projects.")

    for p in projects:
        project_id = p["id"]
        project_name = p.get("name") or p.get("title") or project_id  # some orgs see "title" in responses
        if p.get("status") == "archived" or project_name=='Default project':
            continue

        print(project_name)
        rate_limits = get_project_rate_limits(project_id)
        # Each entry includes: id (rate_limit_id), model, and the current limits
        wanted = [rl for rl in rate_limits if rl.get("model") in TARGET_MODELS]

        if not wanted:
            print(f"- {project_name}: no matching model rate limits found for {TARGET_MODELS} (skipping)")
            continue

        print(f"- {project_name} ({project_id}): updating {len(wanted)} models...")
        for rl in wanted:
            rid = rl["id"]
            model = rl["model"]
            update_rate_limit(project_id, rid, TARGET_RPM, TARGET_TPM)
            print(f"  • {model}: set RPM={TARGET_RPM}, TPM={TARGET_TPM}")
            time.sleep(0.1)  # tiny pause to be polite

    print("Done.")

if __name__ == "__main__":
    main()


OpenAI Admin API key: ··········
Found 40 projects.
amin-fakhar
- amin-fakhar (proj_5Kry8mHoFaG68LYi58Sxzk5m): updating 1 models...
  • gpt-4o-mini: set RPM=120, TPM=100000
maria-rosero
- maria-rosero (proj_N6HDIfF6gkkOsmQWhSVe8Cf6): updating 2 models...
  • gpt-3.5-turbo-0125: set RPM=120, TPM=100000
  • gpt-4o-mini: set RPM=120, TPM=100000
julia-ashikhmin
- julia-ashikhmin (proj_T4FXjoxnL3jSXMI2HjCrpm17): updating 2 models...
  • gpt-3.5-turbo-0125: set RPM=120, TPM=100000
  • gpt-4o-mini: set RPM=120, TPM=100000
michael-baron
- michael-baron (proj_dwFi9aScXTd0gfH0Y9btVDRz): updating 2 models...
  • gpt-3.5-turbo-0125: set RPM=120, TPM=100000
  • gpt-4o-mini: set RPM=120, TPM=100000
addie-ben-joseph
- addie-ben-joseph (proj_XuFWwGNxdtkgjr7sLbcoiwRo): updating 2 models...
  • gpt-3.5-turbo-0125: set RPM=120, TPM=100000
  • gpt-4o-mini: set RPM=120, TPM=100000
mary-boult
- mary-boult (proj_Cu1Php856qbFhxdS1J2gcGsg): updating 2 models...
  • gpt-3.5-turbo-0125: set RPM=120, TPM=100000
 